# Май–июль: Комиссия месяц — MPOS_RENT vs OPER_DOC (без НДС) vs Excel

Эталон — отчёты эквайринга, колонка **«Комиссия (₽ в месяц)»**. Периметр сверки — строки отчёта, ключ `месяц + ИНН + ID договора`.

| Месяц | Файл |
|---|---|
| 2026-05 | `05_Май_2026.xlsx` |
| 2026-06 | `06_Июнь_2026.xlsx` |
| 2026-07 | `07_Июль_2026.xlsx` |

| Источник | Что берём |
|---|---|
| **MPOS_RENT** | `ods_alpha.scd1_mrc_pos_rent.n_amt` — как шаг 10m `final_df` |
| **OPER_DOC** | `ods.scd1_z_R2_IP_DOG_OPER.c_calc_summ` (это и есть OPER_DOC в озере) |
| **OPER_DOC без НДС** | `round(sum(c_calc_summ) / 1.22, 2)` — с этой суммой сравниваем Excel |

Точное совпадение: `|источник − Excel| ≤ 0.01` ₽. `final_df` эта тетрадка не меняет.

Новый kernel нормален. Нужны Impala и три Excel в `/home/jovyan/documents/Equaring/Data`.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_oper_doc_vs_mpos_excel_may_jul'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_MONTHS = ['2026-05', '2026-06', '2026-07']
PERIOD_START = '2026-05-01'
PERIOD_END = '2026-07-31'
PERIOD_END_EXCL = '2026-08-01'
VAT = 1.22
EPS = 0.01
MEM_LIMIT = '8g'
TOP_N = 30
EXAMPLE_START = '2026-04-01'
EXAMPLE_END_EXCL = '2026-09-01'
EXAMPLES_PER_TARIFF = 2
NOTEBOOK_REV = '2026-10-01-oper-doc-july-service-v4'

EXCEL_FILES = {
    '2026-05': {'names': ['05_Май_2026.xlsx'], 'header': 0},
    '2026-06': {'names': ['06_Июнь_2026.xlsx'], 'header': 0},
    '2026-07': {'names': ['07_Июль_2026.xlsx'], 'header': -1},
}
MONTHLY_ALIASES = [
    'Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия в месяц',
    'Комиссия CN (₽ в месяц)', 'commission_monthly', 'Комиссия \n(₽ в месяц)',
]

print('NOTEBOOK_REV:', NOTEBOOK_REV)
print('FOCUS_MONTHS:', FOCUS_MONTHS)
print('OUT_DIR:', OUT_DIR)
print('MPOS = n_amt | OPER_DOC = round(sum(c_calc_summ) / 1.22, 2)')


## 0) Helpers + Impala


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, aliases):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for a in aliases:
        if a in cols:
            return a
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        key = norm(a)
        for nk, orig in nmap.items():
            if key and key in nk:
                return orig
    return None


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
    return df


def is_nz(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).abs() >= EPS


def exact_match(left, right):
    a = pd.to_numeric(left, errors='coerce').fillna(0).round(2)
    b = pd.to_numeric(right, errors='coerce').fillna(0).round(2)
    return np.isclose(a, b, atol=EPS, rtol=0)


def resolve_excel_path(month):
    spec = EXCEL_FILES[month]
    for name in spec['names']:
        p = DATA_DIR / name
        if p.exists():
            return p
    hits = sorted(DATA_DIR.glob(f'*{month[-2:]}*2026*.xlsx'))
    if hits:
        print(f'  fallback Excel {month}: {hits[0].name}')
        return hits[0]
    return None


def read_excel_report(path, header_hint):
    tried = []
    headers = list(range(0, 6)) if header_hint == -1 else [header_hint, *range(0, 6)]
    seen = []
    for h in headers:
        if h in seen:
            continue
        seen.append(h)
        raw = pd.read_excel(path, header=h)
        col_inn = pick_col(raw.columns, ['ИНН', 'inn', 'c_inn'])
        col_agr = pick_col(raw.columns, ['ID договора', 'agr_id', 'abs_agr_id', 'ИД договора'])
        col_mon = pick_col(raw.columns, MONTHLY_ALIASES)
        tried.append((h, col_inn, col_agr, col_mon, list(raw.columns)[:12]))
        if col_inn and col_agr and col_mon:
            return raw, h, col_inn, col_agr, col_mon
    raise RuntimeError(f'Не нашли колонки в {path.name}. tried={tried}')


def classify_row(excel_nz, src_present, src_nz):
    if excel_nz and src_nz:
        return 'оба ≠ 0'
    if (not excel_nz) and (not src_nz):
        return 'оба 0 / нет строки'
    if (not excel_nz) and src_nz:
        return 'источник ≠ 0, отчёт = 0'
    if excel_nz and not src_present:
        return 'отчёт ≠ 0, в источнике нет строки'
    return 'отчёт ≠ 0, источник = 0'


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Эталон: отчёты мая, июня, июля


In [ ]:
excel_parts = []
excel_meta = []
missing = []
for month in FOCUS_MONTHS:
    path = resolve_excel_path(month)
    if path is None:
        missing.append(month)
        print('MISSING Excel for', month, '| DATA_DIR xlsx:', [p.name for p in DATA_DIR.glob('*.xlsx')][:30])
        continue
    header_hint = EXCEL_FILES[month]['header']
    raw_ex, hdr, col_inn, col_agr, col_mon = read_excel_report(path, header_hint)
    col_tar = pick_col(raw_ex.columns, ['Тариф', 'Тарифный план', 'tariff_name'])
    print(f'{month}: {path.name} header={hdr} inn={col_inn} agr={col_agr} monthly={col_mon} rows={len(raw_ex):,}')
    part = pd.DataFrame({
        'report_month': month,
        'inn_key': raw_ex[col_inn].map(normalize_inn_q1),
        'agr_id_key': raw_ex[col_agr].map(normalize_agr_q1),
        'commission_excel': to_num(raw_ex[col_mon]),
        'tariff': raw_ex[col_tar] if col_tar else np.nan,
    })
    part = (
        part.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_excel=('commission_excel', 'max'), tariff=('tariff', 'first'))
    )
    part['commission_excel'] = pd.to_numeric(part['commission_excel'], errors='coerce').fillna(0).round(2)
    part['excel_nz'] = is_nz(part['commission_excel'])
    excel_parts.append(part)
    excel_meta.append({
        'report_month': month,
        'file': path.name,
        'header': hdr,
        'keys': len(part),
        'excel_nz': int(part['excel_nz'].sum()),
        'excel_zero': int((~part['excel_nz']).sum()),
        'sum_excel': float(part['commission_excel'].sum()),
    })

if missing:
    raise FileNotFoundError(
        f'Нет отчётов за {missing}. Положите файлы в {DATA_DIR} '
        f'(05_Май_2026.xlsx, 06_Июнь_2026.xlsx, 07_Июль_2026.xlsx) и перезапустите.'
    )

ex_all = pd.concat(excel_parts, ignore_index=True)
excel_stat = pd.DataFrame(excel_meta)
excel_stat['excel_nz_pct'] = (100.0 * excel_stat['excel_nz'] / excel_stat['keys']).round(2)
print('=== Эталон Excel май–июль ===')
display(excel_stat)
print('keys all months=', f'{len(ex_all):,}', '| unique agr=', f'{ex_all["agr_id_key"].nunique():,}')


## 2) MPOS_RENT: `n_amt` за май–июль


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{PERIOD_START}' as date) and cast('{PERIOD_END}' as date)
),
rent_ranked as (
  select *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num from rent_ranked where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
)
select
  substr(cast(r.d_rent_dt as string), 1, 7) as report_month,
  c.inn_key as inn,
  cast(a.agr_id as string) as agr_id,
  count(*) as rent_rows,
  sum(r.n_amt_num) as commission_mpos
from rent_dedup r
left join terms_active t
  on t.c_nmrc = r.c_nmrc
 and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
left join agreements_active a
  on a.n_agr = t.n_agr
 and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
left join companies_active c
  on c.n_cmp = a.n_cmp_client
group by 1, 2, 3
'''

mpos_raw = fetch_imp(sql_mpos, 'MPOS mapped May-Jul')
mpos = mpos_raw.copy()
mpos['report_month'] = mpos['report_month'].astype(str).str[:7]
mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
mpos['commission_mpos'] = pd.to_numeric(mpos['commission_mpos'], errors='coerce').round(2)
mpos_map = (
    mpos.dropna(subset=['agr_id_key'])
    .query('report_month in @FOCUS_MONTHS')
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(commission_mpos=('commission_mpos', 'sum'), rent_rows=('rent_rows', 'sum'))
)
mpos_map['commission_mpos'] = mpos_map['commission_mpos'].round(2)
mpos_map['mpos_nz'] = is_nz(mpos_map['commission_mpos'])
print('MPOS keys=', f'{len(mpos_map):,}')
display(
    mpos_map.groupby('report_month', as_index=False)
    .agg(keys=('agr_id_key', 'nunique'), nz=('mpos_nz', 'sum'), sum_mpos=('commission_mpos', 'sum'))
    .sort_values('report_month')
)


## 3) OPER_DOC: `sum(c_calc_summ)` и очистка НДС 22%

В озере OPER_DOC — `ods.scd1_z_R2_IP_DOG_OPER`. Месячная комиссия с НДС = `sum(o.c_calc_summ)` на `inn + agr_id`.  
Для сверки с Excel: `commission_oper_net = round(sum(c_calc_summ) / 1.22, 2)`.


In [ ]:
sql_oper = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_calc_summ as double) as c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(o.c_date_create as date) >= cast('{PERIOD_START}' as date)
  and cast(o.c_date_create as date) < cast('{PERIOD_END_EXCL}' as date)
'''
try:
    oper_raw = fetch_imp(sql_oper, 'OPER_DOC DOG_OPER May-Jul')
except Exception as exc:
    print('mixed-case fail, try lowercase:', type(exc).__name__, exc)
    sql_oper = sql_oper.replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper')
    oper_raw = fetch_imp(sql_oper, 'OPER_DOC DOG_OPER May-Jul lc')

oper = oper_raw.copy()
oper['inn_key'] = oper['inn'].map(normalize_inn_q1)
oper['agr_id_key'] = oper['agr_id'].map(normalize_agr_q1)
oper['c_date_create'] = pd.to_datetime(oper['c_date_create'], errors='coerce')
oper['report_month'] = oper['c_date_create'].dt.strftime('%Y-%m')
oper['c_calc_summ'] = pd.to_numeric(oper['c_calc_summ'], errors='coerce')
oper = oper.dropna(subset=['agr_id_key'])
oper = oper.loc[oper['report_month'].isin(FOCUS_MONTHS)].copy()

oper_map = (
    oper.groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(oper_rows=('c_calc_summ', 'size'), commission_oper_gross=('c_calc_summ', 'sum'))
)
oper_map['commission_oper_gross'] = oper_map['commission_oper_gross'].round(2)
oper_map['commission_oper_net'] = (oper_map['commission_oper_gross'] / VAT).round(2)
oper_map['oper_nz'] = is_nz(oper_map['commission_oper_net'])
print('OPER_DOC rows=', f'{len(oper):,}', '| keys=', f'{len(oper_map):,}', '| VAT=', VAT)
display(
    oper_map.groupby('report_month', as_index=False)
    .agg(
        keys=('agr_id_key', 'nunique'),
        nz=('oper_nz', 'sum'),
        sum_gross=('commission_oper_gross', 'sum'),
        sum_net=('commission_oper_net', 'sum'),
    )
    .sort_values('report_month')
)


## 4) Сводка: суммы, покрытие, точное совпадение

Одна строка на месяц и итог за май–июль. Периметр — ключи Excel.

- **Точное, %** — доля строк отчёта, где `|источник − Excel| ≤ 0.01`. Пустой источник считается 0.
- **Оба точны / только MPOS / только OPER_DOC / ни один** — только на строках отчёта с комиссией ≠ 0.
- Покрытие считается отдельно: «нет строки» и «строка есть, сумма 0» не смешиваются.


In [ ]:
keys = ['report_month', 'inn_key', 'agr_id_key']
tri = ex_all.merge(
    mpos_map[keys + ['commission_mpos', 'rent_rows']],
    on=keys,
    how='left',
)
tri = tri.merge(
    oper_map[keys + ['commission_oper_gross', 'commission_oper_net', 'oper_rows']],
    on=keys,
    how='left',
)
tri['mpos_present'] = tri['commission_mpos'].notna()
tri['oper_present'] = tri['commission_oper_gross'].notna()
tri['mpos_nz'] = is_nz(tri['commission_mpos'])
tri['oper_nz'] = is_nz(tri['commission_oper_net'])
tri['exact_mpos'] = exact_match(tri['commission_mpos'], tri['commission_excel'])
tri['exact_oper'] = exact_match(tri['commission_oper_net'], tri['commission_excel'])
tri['delta_mpos'] = tri['commission_mpos'].fillna(0).round(2) - tri['commission_excel']
tri['delta_oper'] = tri['commission_oper_net'].fillna(0).round(2) - tri['commission_excel']
tri['bucket_mpos'] = [
    classify_row(bool(ez), bool(sp), bool(sz))
    for ez, sp, sz in zip(tri['excel_nz'], tri['mpos_present'], tri['mpos_nz'])
]
tri['bucket_oper'] = [
    classify_row(bool(ez), bool(sp), bool(sz))
    for ez, sp, sz in zip(tri['excel_nz'], tri['oper_present'], tri['oper_nz'])
]
tri['three_way'] = np.select(
    [
        ~tri['excel_nz'],
        tri['exact_mpos'] & tri['exact_oper'],
        tri['exact_mpos'] & ~tri['exact_oper'],
        ~tri['exact_mpos'] & tri['exact_oper'],
    ],
    ['отчёт = 0', 'оба точны', 'только MPOS', 'только OPER_DOC'],
    default='ни один',
)

BUCKETS = [
    'оба ≠ 0',
    'оба 0 / нет строки',
    'источник ≠ 0, отчёт = 0',
    'отчёт ≠ 0, в источнике нет строки',
    'отчёт ≠ 0, источник = 0',
]


def month_row(month, frame):
    n = len(frame)
    excel_nz = frame.loc[frame['excel_nz']]
    n_nz = len(excel_nz)
    rec = {
        'report_month': month,
        'n_excel': n,
        'excel_nz': n_nz,
        'sum_excel': float(frame['commission_excel'].sum()),
        'sum_mpos': float(frame['commission_mpos'].fillna(0).sum()),
        'sum_oper_gross': float(frame['commission_oper_gross'].fillna(0).sum()),
        'sum_oper_net': float(frame['commission_oper_net'].fillna(0).sum()),
        'exact_mpos_n': int(frame['exact_mpos'].sum()),
        'exact_oper_n': int(frame['exact_oper'].sum()),
        'nz_both_exact': int((excel_nz['three_way'] == 'оба точны').sum()) if n_nz else 0,
        'nz_only_mpos': int((excel_nz['three_way'] == 'только MPOS').sum()) if n_nz else 0,
        'nz_only_oper': int((excel_nz['three_way'] == 'только OPER_DOC').sum()) if n_nz else 0,
        'nz_neither': int((excel_nz['three_way'] == 'ни один').sum()) if n_nz else 0,
    }
    rec['exact_mpos_pct'] = round(100.0 * rec['exact_mpos_n'] / n, 2) if n else np.nan
    rec['exact_oper_pct'] = round(100.0 * rec['exact_oper_n'] / n, 2) if n else np.nan
    rec['delta_sum_mpos'] = rec['sum_mpos'] - rec['sum_excel']
    rec['delta_sum_oper_net'] = rec['sum_oper_net'] - rec['sum_excel']
    for prefix, col in (('mpos', 'bucket_mpos'), ('oper', 'bucket_oper')):
        vc = frame[col].value_counts()
        rec[f'{prefix}_both_nz'] = int(vc.get('оба ≠ 0', 0))
        rec[f'{prefix}_both_zero'] = int(vc.get('оба 0 / нет строки', 0))
        rec[f'{prefix}_src_nz_excel_0'] = int(vc.get('источник ≠ 0, отчёт = 0', 0))
        rec[f'{prefix}_excel_nz_absent'] = int(vc.get('отчёт ≠ 0, в источнике нет строки', 0))
        rec[f'{prefix}_excel_nz_zero'] = int(vc.get('отчёт ≠ 0, источник = 0', 0))
    return rec


rows = [month_row(month, tri.loc[tri['report_month'] == month]) for month in FOCUS_MONTHS]
rows.append(month_row('2026-05..07', tri))
summary = pd.DataFrame(rows)

show_cols = [
    'report_month', 'n_excel', 'excel_nz',
    'sum_excel', 'sum_mpos', 'sum_oper_gross', 'sum_oper_net',
    'delta_sum_mpos', 'delta_sum_oper_net',
    'exact_mpos_n', 'exact_mpos_pct', 'exact_oper_n', 'exact_oper_pct',
    'nz_both_exact', 'nz_only_mpos', 'nz_only_oper', 'nz_neither',
    'mpos_both_nz', 'mpos_both_zero', 'mpos_src_nz_excel_0', 'mpos_excel_nz_absent', 'mpos_excel_nz_zero',
    'oper_both_nz', 'oper_both_zero', 'oper_src_nz_excel_0', 'oper_excel_nz_absent', 'oper_excel_nz_zero',
]
summary = summary[show_cols]

labels = {
    'report_month': 'Месяц',
    'n_excel': 'Строк Excel',
    'excel_nz': 'Excel ≠ 0',
    'sum_excel': 'Сумма Excel',
    'sum_mpos': 'Сумма MPOS',
    'sum_oper_gross': 'OPER_DOC с НДС',
    'sum_oper_net': 'OPER_DOC без НДС',
    'delta_sum_mpos': 'Δ суммы MPOS',
    'delta_sum_oper_net': 'Δ суммы OPER_DOC net',
    'exact_mpos_n': 'MPOS точно, шт',
    'exact_mpos_pct': 'MPOS точно, %',
    'exact_oper_n': 'OPER_DOC точно, шт',
    'exact_oper_pct': 'OPER_DOC точно, %',
    'nz_both_exact': '≠0 оба точны',
    'nz_only_mpos': '≠0 только MPOS',
    'nz_only_oper': '≠0 только OPER_DOC',
    'nz_neither': '≠0 ни один',
    'mpos_both_nz': 'MPOS оба ≠ 0',
    'mpos_both_zero': 'MPOS оба 0',
    'mpos_src_nz_excel_0': 'MPOS есть, Excel 0',
    'mpos_excel_nz_absent': 'MPOS нет строки',
    'mpos_excel_nz_zero': 'MPOS сумма 0',
    'oper_both_nz': 'OPER оба ≠ 0',
    'oper_both_zero': 'OPER оба 0',
    'oper_src_nz_excel_0': 'OPER есть, Excel 0',
    'oper_excel_nz_absent': 'OPER нет строки',
    'oper_excel_nz_zero': 'OPER сумма 0',
}
print('=== Сводка на периметре Excel ===')
display(summary.rename(columns=labels))

plot_df = summary.loc[summary['report_month'] != '2026-05..07'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(plot_df))
width = 0.36
ax.bar(x - width / 2, plot_df['exact_mpos_pct'], width, label='MPOS_RENT')
ax.bar(x + width / 2, plot_df['exact_oper_pct'], width, label='OPER_DOC без НДС')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['report_month'])
ax.set_ylim(0, 100)
ax.set_ylabel('Точное совпадение, % строк Excel')
ax.set_title('Комиссия месяц: |источник − Excel| ≤ 0.01 ₽')
ax.legend()
fig.tight_layout()
plt.show()

total = summary.loc[summary['report_month'] == '2026-05..07'].iloc[0]
print(
    f"Май–июль точное совпадение: MPOS {total['exact_mpos_pct']}% "
    f"({int(total['exact_mpos_n']):,}/{int(total['n_excel']):,}) | "
    f"OPER_DOC без НДС {total['exact_oper_pct']}% "
    f"({int(total['exact_oper_n']):,}/{int(total['n_excel']):,})"
)
print(
    f"Суммы: Excel {total['sum_excel']:,.2f} | MPOS {total['sum_mpos']:,.2f} "
    f"(Δ {total['delta_sum_mpos']:,.2f}) | OPER_DOC net {total['sum_oper_net']:,.2f} "
    f"(Δ {total['delta_sum_oper_net']:,.2f}) | OPER_DOC gross {total['sum_oper_gross']:,.2f}"
)


## 5) Расхождения

До 30 строк на месяц, самые большие по модулю дельты. В файле — все строки периметра, где хотя бы один источник не совпал с Excel.


In [ ]:
detail_cols = [
    'report_month', 'inn_key', 'agr_id_key', 'tariff',
    'commission_excel', 'commission_mpos', 'commission_oper_gross', 'commission_oper_net',
    'delta_mpos', 'delta_oper', 'exact_mpos', 'exact_oper',
    'bucket_mpos', 'bucket_oper', 'three_way',
]
mismatch = tri.loc[~(tri['exact_mpos'] & tri['exact_oper'])].copy()
mismatch['abs_gap'] = np.maximum(mismatch['delta_mpos'].abs(), mismatch['delta_oper'].abs())
mismatch = mismatch.sort_values(['report_month', 'abs_gap'], ascending=[True, False])

print('Расхождений (не оба источника точны):', f'{len(mismatch):,}', 'из', f'{len(tri):,}')
for month in FOCUS_MONTHS:
    top = mismatch.loc[mismatch['report_month'] == month, detail_cols].head(TOP_N)
    print(f'\\n=== {month}: топ-{TOP_N} по |Δ| ===')
    display(top)

top_export = (
    mismatch.groupby('report_month', group_keys=False)
    .head(TOP_N)[detail_cols]
    .copy()
)


## 6) Примеры для правила месяца: 2 договора на тариф

Коллеги подтверждают, что сумма в OPER_DOC точнее «Комиссии месяц». Платёж бывает разбит на несколько строк, и месяц по `c_date_create` не всегда совпадает с месяцем отчёта.

На каждый тариф из Excel — **2 договора** с ненулевой комиссией хотя бы в одном месяце мая–июля. Сначала те, где в OPER_DOC несколько платежей и несколько разных дат.

По каждому договору:

- Excel за май, июнь, июль и сумма OPER_DOC без НДС, если месяц считать по дате создания;
- все строки OPER_DOC с **1 апреля по 31 августа** (шире отчёта, чтобы видеть платежи на границе месяца), вид комиссии, сумма с НДС и без;
- платежи 1–5 числа или в последние 5 дней месяца помечены как граница месяца.


In [ ]:
def tariff_key(v):
    if pd.isna(v):
        return '(нет тарифа)'
    s = re.sub(r'\s+', ' ', str(v).replace('\xa0', ' ').replace('\n', ' ')).strip()
    if s.lower() in {'', 'nan', 'none', 'nat'}:
        return '(нет тарифа)'
    return s


if 'EXAMPLE_START' not in globals():
    EXAMPLE_START = '2026-04-01'
if 'EXAMPLE_END_EXCL' not in globals():
    EXAMPLE_END_EXCL = '2026-09-01'
if 'EXAMPLES_PER_TARIFF' not in globals():
    EXAMPLES_PER_TARIFF = 2

tri['tariff_key'] = tri['tariff'].map(tariff_key)
contract = (
    tri.groupby(['tariff_key', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(
        excel_nz_months=('excel_nz', 'sum'),
        sum_excel=('commission_excel', 'sum'),
        sum_oper_net=('commission_oper_net', 'sum'),
        sum_abs_delta=('delta_oper', lambda s: float(pd.to_numeric(s, errors='coerce').abs().sum())),
    )
)
pay_stat = (
    oper.groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        pay_rows=('c_date_create', 'size'),
        pay_dates=('c_date_create', 'nunique'),
        pay_months=('report_month', 'nunique'),
    )
)
contract = contract.merge(pay_stat, on=['inn_key', 'agr_id_key'], how='left')
for c in ('pay_rows', 'pay_dates', 'pay_months'):
    contract[c] = contract[c].fillna(0).astype(int)
contract['split'] = contract['pay_rows'] >= 2
candidates = contract.loc[contract['excel_nz_months'] > 0].copy()
candidates = candidates.sort_values(
    ['tariff_key', 'split', 'pay_dates', 'pay_rows', 'sum_abs_delta'],
    ascending=[True, False, False, False, False],
)
picked = candidates.groupby('tariff_key', group_keys=False).head(EXAMPLES_PER_TARIFF).copy()
picked['example_n'] = picked.groupby('tariff_key').cumcount() + 1

tariff_profile = (
    candidates.groupby('tariff_key', as_index=False)
    .agg(
        договоры_с_комиссией=('agr_id_key', 'nunique'),
        из_них_несколько_платежей=('split', 'sum'),
        примеры=('agr_id_key', 'size'),
    )
)
tariff_profile['примеры'] = tariff_profile['tariff_key'].map(picked.groupby('tariff_key').size()).fillna(0).astype(int)
print('Тарифов с ненулевой комиссией Excel:', f'{candidates["tariff_key"].nunique():,}')
print('Примеров:', f'{len(picked):,}', f'(до {EXAMPLES_PER_TARIFF} на тариф)')
display(tariff_profile.sort_values('договоры_с_комиссией', ascending=False))

ids = sorted({str(a) for a in picked['agr_id_key'] if str(a).isdigit()})
if not ids:
    raise RuntimeError('Нет договоров для примеров: проверьте колонку тарифа в Excel и секцию 4.')

oper_table = 'ods.scd1_z_R2_IP_DOG_OPER'
vid_table = 'ods.scd1_z_R2_VID_COMISS'
extra_cols = []
try:
    desc = fetch_imp(f'describe {oper_table}', 'describe DOG_OPER')
except Exception as exc:
    print('describe mixed-case fail:', type(exc).__name__, exc)
    oper_table = 'ods.scd1_z_r2_ip_dog_oper'
    vid_table = 'ods.scd1_z_r2_vid_comiss'
    desc = fetch_imp(f'describe {oper_table}', 'describe DOG_OPER lc')

name_col = desc.columns[0]
raw_names = desc[name_col].astype(str).str.replace('`', '', regex=False)
date_keys = ('date', 'dt', 'period', 'beg', 'end', 'nds', 'vat')
skip_names = {'c_date_create', 'c_calc_summ', 'c_vid_comiss', 'c_parent_id', 'c_parent_class', 'id'}
for name in raw_names:
    token = name.strip().split('.')[-1]
    low = token.lower()
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', token):
        continue
    if low.startswith('ods_') or low in skip_names:
        continue
    if any(k in low for k in date_keys):
        extra_cols.append(token)
extra_cols = extra_cols[:12]
print('Доп. поля OPER_DOC в примерах:', extra_cols or 'нет (только c_date_create и вид комиссии)')

extra_sql = ''.join(f',\n  cast(o.{col} as string) as {col}' for col in extra_cols)
agr_sql = ', '.join(f"'{a}'" for a in ids)
sql_examples = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(m.c_name_in_pr as string) as agreement_num,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_calc_summ as double) as c_calc_summ{extra_sql}
from {oper_table} o
join {vid_table} vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(m.id as string) in ({agr_sql})
  and cast(o.c_date_create as date) >= cast('{EXAMPLE_START}' as date)
  and cast(o.c_date_create as date) < cast('{EXAMPLE_END_EXCL}' as date)
'''
try:
    example_raw = fetch_imp(sql_examples, f'OPER_DOC lines for {len(ids)} examples')
except Exception as exc:
    print('example SQL fail, lowercase tables:', type(exc).__name__, exc)
    sql_examples = (
        sql_examples
        .replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper')
        .replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
    )
    example_raw = fetch_imp(sql_examples, f'OPER_DOC lines lc for {len(ids)} examples')

lines = example_raw.copy() if isinstance(example_raw, pd.DataFrame) else pd.DataFrame()
for col in ['inn', 'agr_id', 'agreement_num', 'commis_type', 'c_date_create', 'c_calc_summ', *extra_cols]:
    if col not in lines.columns:
        lines[col] = pd.NA
lines['inn_key'] = lines['inn'].map(normalize_inn_q1)
lines['agr_id_key'] = lines['agr_id'].map(normalize_agr_q1)
lines['c_date_create'] = pd.to_datetime(lines['c_date_create'], errors='coerce')
lines['c_calc_summ'] = pd.to_numeric(lines['c_calc_summ'], errors='coerce')
lines['pay_month'] = lines['c_date_create'].dt.strftime('%Y-%m')
lines['gross'] = lines['c_calc_summ'].round(2)
lines['net'] = (lines['c_calc_summ'] / VAT).round(2)
day = lines['c_date_create'].dt.day
month_end = lines['c_date_create'].dt.days_in_month
lines['boundary'] = ((day <= 5) | (day >= month_end - 4)).fillna(False)
lines['в_отчёте'] = lines['pay_month'].isin(FOCUS_MONTHS)
lines = lines.merge(
    picked[['tariff_key', 'inn_key', 'agr_id_key', 'example_n']],
    on=['inn_key', 'agr_id_key'],
    how='inner',
)
lines = lines.sort_values(['tariff_key', 'example_n', 'c_date_create', 'commis_type'])

excel_view_parts = []
line_view_parts = []
month_view_parts = []

print('\\n=== Примеры ===')
for tariff, block in picked.groupby('tariff_key', sort=True):
    print('\\n' + '=' * 88)
    print(f'ТАРИФ: {tariff}')
    for rec in block.itertuples(index=False):
        ex = tri.loc[
            (tri['inn_key'] == rec.inn_key) & (tri['agr_id_key'] == rec.agr_id_key)
        ].sort_values('report_month')
        pay = lines.loc[
            (lines['inn_key'] == rec.inn_key) & (lines['agr_id_key'] == rec.agr_id_key)
        ].copy()
        print(
            f"\\n--- пример {int(rec.example_n)} | ИНН {rec.inn_key} | договор {rec.agr_id_key} "
            f"| платежей {len(pay):,} | разных дат {pay['c_date_create'].dt.date.nunique() if len(pay) else 0} ---"
        )
        ex_show = ex[['report_month', 'commission_excel', 'commission_oper_net', 'oper_rows', 'delta_oper']].rename(columns={
            'report_month': 'Месяц',
            'commission_excel': 'Комиссия Excel',
            'commission_oper_net': 'OPER_DOC net по дате создания',
            'oper_rows': 'Платежей в месяце даты',
            'delta_oper': 'Δ net − Excel',
        })
        display(ex_show)
        if pay.empty:
            print('В OPER_DOC с 1 апреля по 31 августа строк нет.')
        else:
            show_cols = [
                'c_date_create', 'pay_month', 'boundary', 'в_отчёте', 'commis_type', 'gross', 'net', 'agreement_num',
            ] + extra_cols
            show_cols = [c for c in show_cols if c in pay.columns]
            pay_show = pay[show_cols].rename(columns={
                'c_date_create': 'Дата создания',
                'pay_month': 'Месяц по дате создания',
                'boundary': 'Граница месяца',
                'в_отчёте': 'Месяц есть в Excel-окне',
                'commis_type': 'Вид комиссии',
                'gross': 'Сумма с НДС',
                'net': 'Сумма без НДС',
                'agreement_num': 'Номер договора',
            })
            display(pay_show)
            by_month = (
                pay.groupby('pay_month', as_index=False)
                .agg(платежей=('net', 'size'), сумма_net=('net', 'sum'), виды=('commis_type', lambda s: ' | '.join(sorted(set(map(str, s))))))
            )
            by_month = by_month.merge(
                ex[['report_month', 'commission_excel']].rename(columns={'report_month': 'pay_month', 'commission_excel': 'Комиссия Excel'}),
                on='pay_month',
                how='left',
            )
            by_month = by_month.rename(columns={'pay_month': 'Месяц', 'сумма_net': 'Сумма net всех видов'})
            display(by_month)

        ex_out = ex[['report_month', 'inn_key', 'agr_id_key', 'tariff_key', 'commission_excel', 'commission_oper_net', 'oper_rows', 'delta_oper']].copy()
        ex_out['example_n'] = int(rec.example_n)
        excel_view_parts.append(ex_out)
        if len(pay):
            keep = pay.copy()
            keep['example_n'] = int(rec.example_n)
            line_view_parts.append(keep)
            month_view_parts.append(by_month.assign(
                tariff_key=tariff,
                inn_key=rec.inn_key,
                agr_id_key=rec.agr_id_key,
                example_n=int(rec.example_n),
            ))

examples_excel = pd.concat(excel_view_parts, ignore_index=True) if excel_view_parts else pd.DataFrame()
examples_lines = pd.concat(line_view_parts, ignore_index=True) if line_view_parts else pd.DataFrame()
examples_by_month = pd.concat(month_view_parts, ignore_index=True) if month_view_parts else pd.DataFrame()
print('\\nПримеры собраны. Excel-строк:', f'{len(examples_excel):,}', '| платежей OPER_DOC:', f'{len(examples_lines):,}')


## 7) Июль, тариф 0: 10 примеров по правилу

Правило, которое проверяем:

- месяц отчёта **июль** = документы OPER_DOC **«Комиссия за совершение операции»** с датой создания в **августе**;
- «Комиссия за обслуживание» не входит;
- верная сумма всегда **без НДС**: `round(sum(c_calc_summ) / 1.22, 2)`, даже если в Excel НДС не сняли.

В сводке два сравнения с одним и тем же правилом: Excel как в файле и Excel / 1.22. Десять договоров — другие, не два разобранных выше. Окно только июль.


In [ ]:
RULE_MONTH = '2026-07'
RULE_PAY_START = '2026-08-01'
RULE_PAY_END_EXCL = '2026-09-01'
RULE_SHOW = 10
RULE_POOL = 80
RULE_NEAR = 0.20
SHOWN_AGR = {'508492576892', '605340034985'}
OP_TYPE_NEEDLE = 'совершение операции'

if 'tariff_key' not in globals():
    def tariff_key(v):
        if pd.isna(v):
            return '(нет тарифа)'
        s = re.sub(r'\s+', ' ', str(v).replace('\xa0', ' ').replace('\n', ' ')).strip()
        if s.lower() in {'', 'nan', 'none', 'nat'}:
            return '(нет тарифа)'
        return s

if 'tariff_key' not in tri.columns:
    tri['tariff_key'] = tri['tariff'].map(tariff_key)

july = tri.loc[tri['report_month'] == RULE_MONTH].copy()
tk = july['tariff_key'].astype(str).str.strip()
tariff_0_mask = tk.str.lower().isin({'0', '0.0'}) | tk.str.match(r'(?i)^0\b')
print('Июль, тариф 0, строк Excel:', int(tariff_0_mask.sum()), '| ≠ 0:', int((tariff_0_mask & july['excel_nz']).sum()))
pool = july.loc[tariff_0_mask & july['excel_nz'] & ~july['agr_id_key'].isin(SHOWN_AGR)].copy()
pool = pool.sort_values('commission_excel', ascending=False).head(RULE_POOL)
ids = sorted({str(a) for a in pool['agr_id_key'] if str(a).isdigit()})
print('Пул для проверки (без двух уже разобранных):', f'{len(ids):,}')
if not ids:
    raise RuntimeError('Нет договоров тарифа 0 за июль с ненулевой комиссией.')

oper_table = globals().get('oper_table', 'ods.scd1_z_R2_IP_DOG_OPER')
vid_table = globals().get('vid_table', 'ods.scd1_z_R2_VID_COMISS')
agr_sql = ', '.join(f"'{a}'" for a in ids)
sql_rule = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_calc_summ as double) as c_calc_summ
from {oper_table} o
join {vid_table} vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(m.id as string) in ({agr_sql})
  and cast(o.c_date_create as date) >= cast('{RULE_PAY_START}' as date)
  and cast(o.c_date_create as date) < cast('{RULE_PAY_END_EXCL}' as date)
'''
try:
    rule_raw = fetch_imp(sql_rule, f'July rule OPER_DOC August lines, n={len(ids)}')
except Exception as exc:
    print('rule SQL fail, lowercase tables:', type(exc).__name__, exc)
    sql_rule = (
        sql_rule
        .replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper')
        .replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
    )
    rule_raw = fetch_imp(sql_rule, f'July rule OPER_DOC August lines lc, n={len(ids)}')

pay = rule_raw.copy() if isinstance(rule_raw, pd.DataFrame) else pd.DataFrame()
for col in ('inn', 'agr_id', 'commis_type', 'c_date_create', 'c_calc_summ'):
    if col not in pay.columns:
        pay[col] = pd.NA
pay['inn_key'] = pay['inn'].map(normalize_inn_q1)
pay['agr_id_key'] = pay['agr_id'].map(normalize_agr_q1)
pay['c_date_create'] = pd.to_datetime(pay['c_date_create'], errors='coerce')
pay['c_calc_summ'] = pd.to_numeric(pay['c_calc_summ'], errors='coerce')
pay['type_norm'] = pay['commis_type'].astype(str).str.replace('\xa0', ' ', regex=False).str.lower()
print('Виды комиссий в августе у пула:')
display(pay['commis_type'].value_counts(dropna=False).rename('rows').reset_index())

op = pay.loc[pay['type_norm'].str.contains(OP_TYPE_NEEDLE, na=False)].copy()
op['gross'] = op['c_calc_summ'].round(2)
op['net'] = (op['c_calc_summ'] / VAT).round(2)

rule_agr = (
    op.groupby('agr_id_key', as_index=False)
    .agg(
        n_payments=('gross', 'size'),
        rule_gross=('gross', 'sum'),
        dates=('c_date_create', lambda s: ', '.join(sorted({d.strftime('%Y-%m-%d') for d in s.dropna()}))),
    )
)
rule_agr['rule_gross'] = rule_agr['rule_gross'].round(2)
rule_agr['rule_net'] = (rule_agr['rule_gross'] / VAT).round(2)

chk = pool.merge(rule_agr, on='agr_id_key', how='left')
chk['n_payments'] = chk['n_payments'].fillna(0).astype(int)
chk['rule_gross'] = chk['rule_gross'].fillna(0).round(2)
chk['rule_net'] = chk['rule_net'].fillna(0).round(2)
chk['excel_net'] = (chk['commission_excel'] / VAT).round(2)
chk['delta_vs_excel'] = (chk['rule_net'] - chk['commission_excel']).round(2)
chk['delta_vs_excel_net'] = (chk['rule_net'] - chk['excel_net']).round(2)
chk['delta_gross_vs_excel'] = (chk['rule_gross'] - chk['commission_excel']).round(2)

def rule_verdict(row):
    if row['n_payments'] == 0:
        return 'нет документа в августе'
    d_net = abs(row['delta_vs_excel'])
    d_vat = min(abs(row['delta_vs_excel_net']), abs(row['delta_gross_vs_excel']))
    if d_net <= d_vat and d_net <= RULE_NEAR:
        return 'Excel уже без НДС' if d_net <= EPS else 'Excel уже без НДС (копейки)'
    if d_vat <= RULE_NEAR:
        return 'в Excel оставили НДС' if d_vat <= EPS else 'в Excel оставили НДС (копейки)'
    return 'не сошлось'

chk['verdict'] = chk.apply(rule_verdict, axis=1)
order = {
    'Excel уже без НДС': 0,
    'Excel уже без НДС (копейки)': 1,
    'в Excel оставили НДС': 2,
    'в Excel оставили НДС (копейки)': 3,
    'не сошлось': 4,
    'нет документа в августе': 5,
}
chk['verdict_ord'] = chk['verdict'].map(order).fillna(9)
chk = chk.sort_values(['verdict_ord', 'commission_excel'], ascending=[True, False])

print('\\n=== Пул июля, тариф 0: правило vs Excel ===')
display(chk['verdict'].value_counts().rename('договоры').reset_index())

picked_parts = []
seen = set()
for _, group in chk.groupby('verdict_ord', sort=True):
    part = group.loc[~group['agr_id_key'].isin(seen)].head(2)
    if len(part):
        picked_parts.append(part)
        seen.update(part['agr_id_key'])
if len(seen) < RULE_SHOW:
    rest = chk.loc[~chk['agr_id_key'].isin(seen)].head(RULE_SHOW - len(seen))
    if len(rest):
        picked_parts.append(rest)
show = pd.concat(picked_parts, ignore_index=True).head(RULE_SHOW) if picked_parts else chk.head(0)
show_cols = show[[
    'inn_key', 'agr_id_key', 'commission_excel', 'excel_net',
    'rule_gross', 'rule_net', 'n_payments', 'dates',
    'delta_vs_excel', 'delta_vs_excel_net', 'verdict',
]].rename(columns={
    'inn_key': 'ИНН',
    'agr_id_key': 'Договор',
    'commission_excel': 'Excel июль',
    'excel_net': 'Excel / 1.22',
    'rule_gross': 'Август совершение с НДС',
    'rule_net': 'Правило: без НДС',
    'n_payments': 'Платежей',
    'dates': 'Даты создания',
    'delta_vs_excel': 'Δ правило − Excel',
    'delta_vs_excel_net': 'Δ правило − Excel/1.22',
    'verdict': 'Сверка',
})
print(f'\\n=== {RULE_SHOW} примеров, июль, тариф 0 ===')
display(show_cols)

show_ids = set(show['agr_id_key'])
line_show = op.loc[op['agr_id_key'].isin(show_ids), [
    'inn_key', 'agr_id_key', 'c_date_create', 'commis_type', 'gross', 'net',
]].sort_values(['agr_id_key', 'c_date_create']).rename(columns={
    'inn_key': 'ИНН',
    'agr_id_key': 'Договор',
    'c_date_create': 'Дата создания',
    'commis_type': 'Вид комиссии',
    'gross': 'Сумма с НДС',
    'net': 'Сумма без НДС',
})
print('\\nПлатежи «совершение операции» за август у этих 10 договоров')
display(line_show)

july_rule_check = show_cols.copy()
july_rule_lines = line_show.copy()
july_rule_pool = chk[[
    'inn_key', 'agr_id_key', 'commission_excel', 'excel_net',
    'rule_gross', 'rule_net', 'n_payments', 'dates', 'verdict',
]].copy()


## 8) Июль: обслуживание там, где нет «совершения», и договор 373501537273

Для договоров тарифа 0 из пула, у которых в августе нет «Комиссии за совершение операции», смотрим августовскую **«Комиссию за обслуживание»**. Сумма без НДС — `round(sum / 1.22, 2)`. Рядом Excel июля и Excel / 1.22.

Отдельно договор **373501537273**: все строки OPER_DOC с датой создания с 1 мая по 31 августа, оба вида комиссии, и Excel за май–июль. Так видно, не собрана ли августовская сумма из более ранних доначислений.


In [ ]:
SVC_NEEDLE = 'обслуживан'
PROBE_AGR = '373501537273'
PROBE_START = '2026-05-01'
PROBE_END_EXCL = '2026-09-01'

if 'pay' not in globals() or 'chk' not in globals():
    raise RuntimeError('Сначала выполните секцию 7: нужны pay и chk.')

svc = pay.loc[pay['type_norm'].str.contains(SVC_NEEDLE, na=False)].copy()
svc['gross'] = svc['c_calc_summ'].round(2)
svc['net'] = (svc['c_calc_summ'] / VAT).round(2)
svc_agr = (
    svc.groupby('agr_id_key', as_index=False)
    .agg(
        n_svc=('gross', 'size'),
        svc_gross=('gross', 'sum'),
        dates=('c_date_create', lambda s: ', '.join(sorted({d.strftime('%Y-%m-%d') for d in s.dropna()}))),
    )
)
svc_agr['svc_gross'] = svc_agr['svc_gross'].round(2)
svc_agr['svc_net'] = (svc_agr['svc_gross'] / VAT).round(2)

missing = chk.loc[chk['verdict'] == 'нет документа в августе'].copy()
svc_chk = missing.merge(svc_agr, on='agr_id_key', how='left')
svc_chk['n_svc'] = svc_chk['n_svc'].fillna(0).astype(int)
svc_chk['svc_gross'] = svc_chk['svc_gross'].fillna(0).round(2)
svc_chk['svc_net'] = svc_chk['svc_net'].fillna(0).round(2)
svc_chk['delta_vs_excel'] = (svc_chk['svc_net'] - svc_chk['commission_excel']).round(2)
svc_chk['delta_vs_excel_net'] = (svc_chk['svc_net'] - svc_chk['excel_net']).round(2)
svc_chk['delta_gross_vs_excel'] = (svc_chk['svc_gross'] - svc_chk['commission_excel']).round(2)

def svc_verdict(row):
    if row['n_svc'] == 0:
        return 'нет обслуживания в августе'
    d_net = abs(row['delta_vs_excel'])
    d_vat = min(abs(row['delta_vs_excel_net']), abs(row['delta_gross_vs_excel']))
    if d_net <= d_vat and d_net <= RULE_NEAR:
        return 'Excel уже без НДС' if d_net <= EPS else 'Excel уже без НДС (копейки)'
    if d_vat <= RULE_NEAR:
        return 'в Excel оставили НДС' if d_vat <= EPS else 'в Excel оставили НДС (копейки)'
    return 'не сошлось'

svc_chk['verdict_svc'] = svc_chk.apply(svc_verdict, axis=1)
print('Договоры без августовского «совершения»:', f'{len(svc_chk):,}')
print('=== Августовское обслуживание vs Excel июля ===')
display(svc_chk['verdict_svc'].value_counts().rename('договоры').reset_index())

svc_view = svc_chk.sort_values(['verdict_svc', 'commission_excel'], ascending=[True, False])[[
    'inn_key', 'agr_id_key', 'commission_excel', 'excel_net',
    'svc_gross', 'svc_net', 'n_svc', 'dates',
    'delta_vs_excel', 'delta_vs_excel_net', 'verdict_svc',
]].rename(columns={
    'inn_key': 'ИНН',
    'agr_id_key': 'Договор',
    'commission_excel': 'Excel июль',
    'excel_net': 'Excel / 1.22',
    'svc_gross': 'Обслуживание с НДС',
    'svc_net': 'Обслуживание без НДС',
    'n_svc': 'Платежей',
    'dates': 'Даты создания',
    'delta_vs_excel': 'Δ без НДС − Excel',
    'delta_vs_excel_net': 'Δ без НДС − Excel/1.22',
    'verdict_svc': 'Сверка',
})
matched = svc_view.loc[~svc_view['Сверка'].isin(['не сошлось', 'нет обслуживания в августе'])]
print('\\nГде обслуживание объясняет Excel июля:', f'{len(matched):,}')
display(matched)
print('\\nГде не объясняет, первые 15 по Excel:')
display(svc_view.loc[svc_view['Сверка'].isin(['не сошлось', 'нет обслуживания в августе'])].head(15))

july_service_fee = svc_view.copy()

oper_table = globals().get('oper_table', 'ods.scd1_z_R2_IP_DOG_OPER')
vid_table = globals().get('vid_table', 'ods.scd1_z_R2_VID_COMISS')
sql_probe = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_calc_summ as double) as c_calc_summ
from {oper_table} o
join {vid_table} vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(m.id as string) = '{PROBE_AGR}'
  and cast(o.c_date_create as date) >= cast('{PROBE_START}' as date)
  and cast(o.c_date_create as date) < cast('{PROBE_END_EXCL}' as date)
'''
try:
    probe_raw = fetch_imp(sql_probe, f'OPER_DOC {PROBE_AGR} May-Aug')
except Exception as exc:
    print('probe SQL fail, lowercase tables:', type(exc).__name__, exc)
    sql_probe = (
        sql_probe
        .replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper')
        .replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
    )
    probe_raw = fetch_imp(sql_probe, f'OPER_DOC {PROBE_AGR} May-Aug lc')

probe = probe_raw.copy() if isinstance(probe_raw, pd.DataFrame) else pd.DataFrame()
for col in ('inn', 'agr_id', 'commis_type', 'c_date_create', 'c_calc_summ'):
    if col not in probe.columns:
        probe[col] = pd.NA
probe['c_date_create'] = pd.to_datetime(probe['c_date_create'], errors='coerce')
probe['c_calc_summ'] = pd.to_numeric(probe['c_calc_summ'], errors='coerce')
probe['pay_month'] = probe['c_date_create'].dt.strftime('%Y-%m')
probe['gross'] = probe['c_calc_summ'].round(2)
probe['net'] = (probe['c_calc_summ'] / VAT).round(2)
probe = probe.sort_values(['c_date_create', 'commis_type'])

excel_probe = tri.loc[tri['agr_id_key'] == PROBE_AGR, [
    'report_month', 'inn_key', 'tariff_key', 'commission_excel',
]].sort_values('report_month').copy()
excel_probe['excel_net'] = (excel_probe['commission_excel'] / VAT).round(2)

print(f'\\n=== Договор {PROBE_AGR}: Excel май–июль ===')
display(excel_probe.rename(columns={
    'report_month': 'Месяц',
    'inn_key': 'ИНН',
    'tariff_key': 'Тариф',
    'commission_excel': 'Excel',
    'excel_net': 'Excel / 1.22',
}))
print(f'Строки OPER_DOC {PROBE_START} .. {PROBE_END_EXCL}, оба вида:')
display(probe[[
    'c_date_create', 'pay_month', 'commis_type', 'gross', 'net',
]].rename(columns={
    'c_date_create': 'Дата создания',
    'pay_month': 'Месяц даты',
    'commis_type': 'Вид комиссии',
    'gross': 'Сумма с НДС',
    'net': 'Сумма без НДС',
}))

by_type = (
    probe.groupby(['pay_month', 'commis_type'], as_index=False)
    .agg(платежей=('gross', 'size'), сумма_с_НДС=('gross', 'sum'), сумма_без_НДС=('net', 'sum'))
)
by_type = by_type.merge(
    excel_probe[['report_month', 'commission_excel', 'excel_net']].rename(columns={
        'report_month': 'pay_month',
        'commission_excel': 'Excel этого месяца',
        'excel_net': 'Excel/1.22 этого месяца',
    }),
    on='pay_month',
    how='left',
)
print('\\nСуммы по месяцу даты создания:')
display(by_type.rename(columns={'pay_month': 'Месяц даты', 'commis_type': 'Вид комиссии'}))

op_probe = probe.loc[probe['commis_type'].astype(str).str.lower().str.contains('совершение операции', na=False)]
print(
    'Совершение за май–август: с НДС',
    f"{op_probe['gross'].sum():,.2f}",
    '| без НДС',
    f"{op_probe['net'].sum():,.2f}",
)
print(
    'Excel май+июнь+июль:',
    f"{excel_probe['commission_excel'].sum():,.2f}",
    '| Excel/1.22:',
    f"{excel_probe['excel_net'].sum():,.2f}",
)

probe_3735_lines = probe.copy()
probe_3735_by_month = by_type.copy()
july_service_fee = svc_view.copy()


## 9) Выгрузка


In [ ]:
out_xlsx = OUT_DIR / 'oper_doc_vs_mpos_excel_2026_05_07.xlsx'
out_csv = OUT_DIR / 'oper_doc_vs_mpos_excel_2026_05_07_keys.csv'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    excel_stat.to_excel(w, sheet_name='excel_stat', index=False)
    summary.to_excel(w, sheet_name='summary', index=False)
    top_export.to_excel(w, sheet_name='top_mismatches', index=False)
    if 'tariff_profile' in globals() and isinstance(tariff_profile, pd.DataFrame):
        tariff_profile.to_excel(w, sheet_name='tariff_profile', index=False)
    if 'examples_excel' in globals() and isinstance(examples_excel, pd.DataFrame) and len(examples_excel):
        examples_excel.to_excel(w, sheet_name='examples_excel', index=False)
    if 'examples_by_month' in globals() and isinstance(examples_by_month, pd.DataFrame) and len(examples_by_month):
        examples_by_month.to_excel(w, sheet_name='examples_by_month', index=False)
    if 'examples_lines' in globals() and isinstance(examples_lines, pd.DataFrame) and len(examples_lines):
        examples_lines.to_excel(w, sheet_name='examples_oper_lines', index=False)
    if 'july_rule_check' in globals() and isinstance(july_rule_check, pd.DataFrame) and len(july_rule_check):
        july_rule_check.to_excel(w, sheet_name='july_rule_10', index=False)
    if 'july_rule_lines' in globals() and isinstance(july_rule_lines, pd.DataFrame) and len(july_rule_lines):
        july_rule_lines.to_excel(w, sheet_name='july_rule_lines', index=False)
    if 'july_rule_pool' in globals() and isinstance(july_rule_pool, pd.DataFrame) and len(july_rule_pool):
        july_rule_pool.to_excel(w, sheet_name='july_rule_pool', index=False)
    if 'july_service_fee' in globals() and isinstance(july_service_fee, pd.DataFrame) and len(july_service_fee):
        july_service_fee.to_excel(w, sheet_name='july_service_fee', index=False)
    if 'probe_3735_lines' in globals() and isinstance(probe_3735_lines, pd.DataFrame) and len(probe_3735_lines):
        probe_3735_lines.to_excel(w, sheet_name='agr_373501537273', index=False)
    if 'probe_3735_by_month' in globals() and isinstance(probe_3735_by_month, pd.DataFrame) and len(probe_3735_by_month):
        probe_3735_by_month.to_excel(w, sheet_name='agr_373501537273_m', index=False)
tri[detail_cols].to_csv(out_csv, index=False)
print('Saved:', out_xlsx)
print('Saved:', out_csv, '| rows=', f'{len(tri):,}')
print('NOTEBOOK_REV', NOTEBOOK_REV)
print('final_df не менялся. OPER_DOC net = round(sum(c_calc_summ) / 1.22, 2)')
